# 06 — Causal Inference: Media Effectiveness Analysis

## Objective
Estimate the **causal effect** of each media channel's spend on quote requests,
controlling for seasonality, weather, and holidays.

This is **Step 1** of the professor's two-step framework:
1. *Causal inference* — which channels drive quotes, and at what cost? (this notebook)
2. *Budget optimization* — given those estimates, how should the budget be allocated? (notebook 07)

### Modelling Strategy
- **3 separate models** — one per product category (above-ground pools, in-ground pools, spas) to capture heterogeneous media response across customer segments.
- **Ridge regression** as the primary estimator (handles multicollinearity with limited observations).
- **OLS** as a baseline; **Elastic Net** for variable-selection check.
- **Adstock + Hill saturation** applied inline (self-contained notebook).
- **Bootstrap** for uncertainty quantification (1 000 resamples).
- **Robustness checks** including sensitivity to adstock/saturation assumptions.

### Data Support
- **Now includes Budget 2023, 2024, and 2025 data**
- Works dynamically with any date range in the processed data
- No hardcoded year filters - automatically adapts to available observations
- Handles data gaps between fiscal years appropriately via geometric decay

### Key Deliverable
A per-product effectiveness table: which media channels drive which product categories, and at what cost per incremental quote.

---
## Section 0 — Setup & Data Loading

In [ ]:
# ---------- imports ----------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings, json

from sklearn.linear_model import Ridge, RidgeCV, ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings('ignore')

# ---------- plot settings ----------
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize': (12, 6), 'font.size': 10,
                      'figure.dpi': 120, 'savefig.bbox': 'tight'})
sns.set_palette('husl')

# ---------- paths ----------
project_root = Path().cwd().parent if Path().cwd().name == 'notebooks' else Path().cwd()
processed_path = project_root / 'data' / 'processed'
figures_path   = project_root / 'reports' / 'figures'
figures_path.mkdir(parents=True, exist_ok=True)

print(f'Project root : {project_root}')
print(f'Processed    : {processed_path}')
print(f'Figures      : {figures_path}')

In [ ]:
# ---------- load data ----------
df_raw = pd.read_csv(processed_path / 'quotes_spend_weather.csv')
df_raw['date'] = pd.to_datetime(df_raw['date'])
df_raw = df_raw.sort_values('date').reset_index(drop=True)

print(f'Shape : {df_raw.shape}')
print(f'Date range : {df_raw["date"].min().date()} — {df_raw["date"].max().date()}')
print(f'\nTarget summary:')
for col in ['piscines_hors_terre', 'piscines_creusees', 'spas']:
    s = df_raw[col]
    print(f'  {col:25s}  mean={s.mean():.0f}  std={s.std():.0f}  min={s.min():.0f}  max={s.max():.0f}')

---
## Section 1 — Data Preparation

1. Aggregate 15 raw spend columns → 7 channel groups.
2. Adstock transformation (geometric decay).
3. Hill saturation.
4. Build control features (seasonality, weather, holidays).

In [ ]:
# ===================================================================
# 1-A  CHANNEL AGGREGATION  (15 raw → 7 groups)
# ===================================================================

df = df_raw.copy()

CHANNEL_MAP = {
    'television': ['spend_television'],
    'radio': ['spend_radio', 'spend_radio_numérique'],
    'panneaux': ['spend_panneaux', 'spend_panneaux_et_affichages_numériques'],
    'social_media': ['spend_social_media'],
    'preroll': ['spend_preroll___premium'],
    'premium_display': [
        'spend_bannières_web___premium',
        'spend_google_ads',
        'spend_lapresse_(lp+,_preroll,_display)',
        'spend_contenu_de_marque',
    ],
    'circulaire_digital': ['spend_circulaire_digital', 'spend_circulaire_digitale'],
}

MEDIA_CHANNELS = []
for group, cols in CHANNEL_MAP.items():
    col_name = f'media_{group}'
    existing = [c for c in cols if c in df.columns]
    df[col_name] = df[existing].sum(axis=1) if existing else 0
    MEDIA_CHANNELS.append(col_name)

df['media_total'] = df[MEDIA_CHANNELS].sum(axis=1)

print('Channel groups (7):')
for ch in MEDIA_CHANNELS:
    total = df[ch].sum()
    pct = total / df['media_total'].sum() * 100
    nz  = (df[ch] > 0).sum()
    print(f'  {ch:30s}  ${total:>12,.0f}  ({pct:5.1f}%)  non-zero months: {nz}')
print(f'  {"TOTAL":30s}  ${df["media_total"].sum():>12,.0f}')

In [ ]:
# ===================================================================
# 1-B  ADSTOCK TRANSFORMATION  (geometric decay)
# ===================================================================

# Load calibrated parameters from notebook 05
params_path = processed_path / 'optimal_transformation_params.json'
with open(params_path) as f:
    CAUSAL_PARAMS = json.load(f)

DECAY_RATES = CAUSAL_PARAMS['decay_rates']

print(f'Parameters loaded from: {params_path.name}')
print(f'Calibration method: {CAUSAL_PARAMS["calibration_method"]}')
print(f'Calibration date: {CAUSAL_PARAMS["calibration_date"]}')
print()

# Import transformation functions from shared utilities
import sys
sys.path.append(str(project_root))
from src.features.transformations import geometric_adstock, hill_saturation, hill_derivative

# Note on data gaps:
# The dataset may have gaps (e.g., missing months between fiscal years).
# The geometric adstock transformation handles this appropriately - the decay
# naturally shrinks the carryover effect over any gap period.
for ch in MEDIA_CHANNELS:
    lam = DECAY_RATES[ch]
    df[f'{ch}_adstock'] = geometric_adstock(df[ch].fillna(0).values, lam)

ADSTOCK_COLS = [f'{ch}_adstock' for ch in MEDIA_CHANNELS]

print('Adstock columns created (with decay rates from NB05):')
for ch in MEDIA_CHANNELS:
    lam = DECAY_RATES[ch]
    source = CAUSAL_PARAMS.get('decay_rate_sources', {}).get(ch, 'unknown')
    hl  = np.log(0.5) / np.log(lam) if lam > 0 else 0
    print(f'  {ch}_adstock   λ={lam}  half-life={hl:.1f} months  ({source})')

In [ ]:
# ===================================================================
# 1-C  HILL SATURATION  (parameters from NB05)
# ===================================================================

SATURATION_PARAMS = {}
HILL_ALPHA = 2  # steepness – same for all channels

for ch in MEDIA_CHANNELS:
    adstock_col = f'{ch}_adstock'
    sat_params = CAUSAL_PARAMS['saturation_params'].get(ch, {})
    K = sat_params.get('K', None)
    
    # If K not available from NB05, compute from data
    if K is None:
        nz = df[adstock_col][df[adstock_col] > 0]
        K = float(nz.median()) if len(nz) > 0 else 1.0
    
    alpha = sat_params.get('alpha', HILL_ALPHA)
    SATURATION_PARAMS[ch] = {'K': K, 'alpha': alpha}
    df[f'{ch}_saturated'] = hill_saturation(df[adstock_col].values, K, alpha)

SATURATED_COLS = [f'{ch}_saturated' for ch in MEDIA_CHANNELS]

print('Saturation parameters (Hill, from NB05):')
for ch, p in SATURATION_PARAMS.items():
    print(f'  {ch:30s}  K=${p["K"]:>12,.0f}   alpha={p["alpha"]}')

In [ ]:
# ===================================================================
# 1-D  CONTROL FEATURES
# ===================================================================

# Fourier seasonality (period = 12 months, 1 harmonic pair)
df['sin_1'] = np.sin(2 * np.pi * df['month_num'] / 12)
df['cos_1'] = np.cos(2 * np.pi * df['month_num'] / 12)

# Weather — standardised avg_temp_mean
temp_mean = df['avg_temp_mean'].mean()
temp_std  = df['avg_temp_mean'].std()
df['avg_temp_mean_scaled'] = (df['avg_temp_mean'] - temp_mean) / temp_std

# Binary pool season (Apr–Aug)
df['is_pool_season'] = df['month_num'].isin([4, 5, 6, 7, 8]).astype(int)

CONTROL_COLS = ['sin_1', 'cos_1', 'avg_temp_mean_scaled', 'is_pool_season']

TARGET_COLS = ['piscines_hors_terre', 'piscines_creusees', 'spas']
PRODUCT_LABELS = {
    'piscines_hors_terre': 'Above-Ground Pools',
    'piscines_creusees':   'In-Ground Pools',
    'spas':                'Spas',
}

# Final feature matrix
FEATURE_COLS = SATURATED_COLS + CONTROL_COLS

print(f'Features ({len(FEATURE_COLS)}):')
for f in FEATURE_COLS:
    print(f'  {f}')
print(f'\nTargets: {TARGET_COLS}')
print(f'Observations: {len(df)}')

---
## Section 2 — Identification Challenge (Pre-Modelling Diagnostics)

Why naive correlation ≠ causation in this dataset:
- **Seasonality confound** — both spending and demand peak in spring/summer.
- **Multicollinearity** — most channels fire together (campaign bursts).
- **Small N** — 19 observations with 11 parameters is tight.

In [ ]:
# ---------- Correlation matrix: saturated media + targets ----------
corr_cols = TARGET_COLS + SATURATED_COLS
corr_mat  = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr_mat, dtype=bool))
labels = [c.replace('media_', '').replace('_saturated', '') for c in corr_cols]
sns.heatmap(corr_mat, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, xticklabels=labels, yticklabels=labels, square=True)
ax.set_title('Correlation: Saturated Media Features vs Quote Targets')
plt.savefig(figures_path / 'causal_correlation_matrix.png', dpi=150)
plt.show()

In [ ]:
# ---------- VIF analysis ----------
X_vif = df[FEATURE_COLS].copy()
X_vif = X_vif.fillna(0)
# Add constant for VIF calc
X_vif_c = sm.add_constant(X_vif)

vif_data = pd.DataFrame({
    'Feature': X_vif_c.columns[1:],
    'VIF': [variance_inflation_factor(X_vif_c.values, i+1) for i in range(len(FEATURE_COLS))]
}).sort_values('VIF', ascending=False)

print('Variance Inflation Factors (VIF > 5 indicates concern):')
print(vif_data.to_string(index=False))

In [ ]:
# ---------- Confounding wedge: spend, quotes, temperature ----------
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)

ax0 = axes[0]
ax0.bar(df['date'], df['media_total'], width=25, color='steelblue', alpha=0.7)
ax0.set_ylabel('Total Media Spend ($)')
ax0.set_title('Confounding Wedge: Spend, Quotes, and Temperature Move Together')

ax1 = axes[1]
for t in TARGET_COLS:
    ax1.plot(df['date'], df[t], marker='o', label=PRODUCT_LABELS[t])
ax1.set_ylabel('Quote Requests')
ax1.legend(loc='upper left')

ax2 = axes[2]
ax2.plot(df['date'], df['avg_temp_mean'], color='orangered', marker='s')
ax2.set_ylabel('Avg Temperature (°C)')
ax2.set_xlabel('Month')

plt.tight_layout()
plt.savefig(figures_path / 'causal_confounding_wedge.png', dpi=150)
plt.show()

**Take-away:** All three series rise and fall together.  
Naive correlation of spend with quotes would overstate the true media effect because warm weather independently drives both ad budgets and consumer demand.  
Ridge regression with weather and seasonality controls is our main strategy to disentangle media impact from the seasonal confound.

---
## Section 3 — Model 1: OLS Baseline (per product)

In [ ]:
# ---------- OLS per product ----------
X_ols = sm.add_constant(df[FEATURE_COLS].fillna(0))

ols_results = {}
for target in TARGET_COLS:
    y = df[target].values
    model = sm.OLS(y, X_ols).fit()
    ols_results[target] = model
    print('=' * 72)
    print(f'OLS — {PRODUCT_LABELS[target]}')
    print('=' * 72)
    print(model.summary2())
    print()

In [ ]:
# ---------- OLS diagnostic plots ----------
fig, axes = plt.subplots(len(TARGET_COLS), 3, figsize=(16, 4 * len(TARGET_COLS)))

for row, target in enumerate(TARGET_COLS):
    model = ols_results[target]
    resid = model.resid
    fitted = model.fittedvalues

    # Residuals vs fitted
    ax = axes[row, 0]
    ax.scatter(fitted, resid, alpha=0.7)
    ax.axhline(0, color='red', linestyle='--', linewidth=0.8)
    ax.set_xlabel('Fitted'); ax.set_ylabel('Residuals')
    ax.set_title(f'{PRODUCT_LABELS[target]} — Residuals vs Fitted')

    # Q-Q plot
    ax = axes[row, 1]
    sm.qqplot(resid, line='45', ax=ax, alpha=0.7)
    ax.set_title(f'{PRODUCT_LABELS[target]} — Q-Q Plot')

    # Residuals over time
    ax = axes[row, 2]
    ax.plot(df['date'], resid, marker='o')
    ax.axhline(0, color='red', linestyle='--', linewidth=0.8)
    ax.set_xlabel('Date'); ax.set_ylabel('Residuals')
    ax.set_title(f'{PRODUCT_LABELS[target]} — Residuals Over Time')

plt.tight_layout()
plt.savefig(figures_path / 'causal_ols_diagnostics.png', dpi=150)
plt.show()

print('\nOLS Limitation: p/N ratio is ~11/19 — overfitting risk is high.\n'
      'Wide confidence intervals and unstable coefficients expected.\n'
      'Ridge regression mitigates this via shrinkage.')

---
## Section 4 — Model 2: Ridge Regression (Primary Model, per product)

Ridge shrinks coefficients toward zero, stabilising estimates when
predictors are correlated and N is small.  
We select the regularisation parameter α via **leave-one-out cross-validation** (LOOCV),
which is the natural choice for N = 19.

In [ ]:
# ---------- Ridge with LOOCV alpha selection ----------
np.random.seed(42)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[FEATURE_COLS].fillna(0))

ALPHAS = np.logspace(-2, 3, 100)  # 0.01 to 1000

ridge_results = {}  # target -> dict of model, coefs, alpha, etc.

for target in TARGET_COLS:
    y = df[target].values

    # LOOCV for alpha selection
    ridge_cv = RidgeCV(alphas=ALPHAS, scoring='neg_mean_squared_error',
                       cv=LeaveOneOut())
    ridge_cv.fit(X_scaled, y)
    best_alpha = ridge_cv.alpha_

    # Refit with best alpha
    model = Ridge(alpha=best_alpha)
    model.fit(X_scaled, y)

    y_pred = model.predict(X_scaled)
    r2     = r2_score(y, y_pred)
    mae    = mean_absolute_error(y, y_pred)

    # Store results
    ridge_results[target] = {
        'model':      model,
        'best_alpha': best_alpha,
        'r2':         r2,
        'mae':        mae,
        'y_pred':     y_pred,
    }

    print(f'{PRODUCT_LABELS[target]:25s}  alpha={best_alpha:>9.2f}   '
          f'R²={r2:.3f}   MAE={mae:.1f}')

In [ ]:
# ---------- Bootstrap for coefficient confidence intervals ----------
N_BOOT = 1000
boot_coefs = {t: [] for t in TARGET_COLS}

n = len(df)
rng = np.random.RandomState(42)

for b in range(N_BOOT):
    idx = rng.choice(n, size=n, replace=True)
    X_b = X_scaled[idx]
    for target in TARGET_COLS:
        y_b = df[target].values[idx]
        alpha_b = ridge_results[target]['best_alpha']
        m = Ridge(alpha=alpha_b).fit(X_b, y_b)
        boot_coefs[target].append(m.coef_)

# Summarise bootstrap
boot_summary = {}
for target in TARGET_COLS:
    arr = np.array(boot_coefs[target])  # (N_BOOT, n_features)
    boot_summary[target] = pd.DataFrame({
        'feature':  FEATURE_COLS,
        'coef':     ridge_results[target]['model'].coef_,
        'boot_mean': arr.mean(axis=0),
        'ci_lower': np.percentile(arr, 5,  axis=0),
        'ci_upper': np.percentile(arr, 95, axis=0),
    })

print('Bootstrap done (1 000 resamples).  90 % CI computed.')

In [ ]:
# ---------- Coefficient bar charts with CI ----------
fig, axes = plt.subplots(1, 3, figsize=(18, 7), sharey=False)

for idx, target in enumerate(TARGET_COLS):
    ax = axes[idx]
    bs = boot_summary[target]
    # Show only media features
    media_mask = bs['feature'].str.contains('_saturated')
    bs_media = bs[media_mask].copy()
    bs_media['label'] = bs_media['feature'].str.replace('media_', '').str.replace('_saturated', '')
    bs_media = bs_media.sort_values('coef')

    colors = ['#e74c3c' if c < 0 else '#2ecc71' for c in bs_media['coef']]
    ax.barh(bs_media['label'], bs_media['coef'], color=colors, alpha=0.8)
    ax.errorbar(bs_media['coef'], bs_media['label'],
                xerr=[bs_media['coef'] - bs_media['ci_lower'],
                      bs_media['ci_upper'] - bs_media['coef']],
                fmt='none', ecolor='black', capsize=3, linewidth=1)
    ax.axvline(0, color='grey', linestyle='--', linewidth=0.8)
    ax.set_title(f'{PRODUCT_LABELS[target]}', fontsize=12)
    ax.set_xlabel('Ridge Coefficient (standardised scale)')

axes[0].set_ylabel('Media Channel')
fig.suptitle('Ridge Regression Coefficients with 90 % Bootstrap CI', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(figures_path / 'causal_ridge_coefficients.png', dpi=150)
plt.show()

In [ ]:
# ---------- Actual vs predicted ----------
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, target in enumerate(TARGET_COLS):
    ax = axes[idx]
    y_true = df[target].values
    y_pred = ridge_results[target]['y_pred']
    ax.scatter(y_true, y_pred, alpha=0.7, edgecolors='k', linewidths=0.5)
    mn, mx = min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())
    ax.plot([mn, mx], [mn, mx], 'r--', linewidth=0.8)
    ax.set_xlabel('Actual Quotes')
    ax.set_ylabel('Predicted Quotes')
    r2 = ridge_results[target]['r2']
    ax.set_title(f'{PRODUCT_LABELS[target]}  (R²={r2:.3f})')

fig.suptitle('Ridge — Actual vs Predicted', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(figures_path / 'causal_ridge_actual_vs_pred.png', dpi=150)
plt.show()

In [ ]:
# ---------- Media contribution decomposition ----------
fig, axes = plt.subplots(len(TARGET_COLS), 1, figsize=(14, 5 * len(TARGET_COLS)), sharex=True)

for row, target in enumerate(TARGET_COLS):
    ax = axes[row]
    model = ridge_results[target]['model']
    intercept = model.intercept_

    # Contributions from each feature (in original quote units)
    contribs = X_scaled * model.coef_  # (n, p)
    media_idx = [FEATURE_COLS.index(c) for c in SATURATED_COLS]
    ctrl_idx  = [FEATURE_COLS.index(c) for c in CONTROL_COLS]

    # Stack media contributions
    media_contribs = pd.DataFrame(
        contribs[:, media_idx],
        columns=[c.replace('media_', '').replace('_saturated', '') for c in SATURATED_COLS]
    )
    # Clip negatives to zero for stacking (show net below)
    media_pos = media_contribs.clip(lower=0)
    control_contrib = contribs[:, ctrl_idx].sum(axis=1)

    ax.fill_between(df['date'], 0, intercept, alpha=0.3, label='Baseline (intercept)', color='grey')
    bottom = np.full(len(df), intercept)
    for col in media_pos.columns:
        vals = media_pos[col].values
        ax.fill_between(df['date'], bottom, bottom + vals, alpha=0.6, label=col.title())
        bottom = bottom + vals
    ax.fill_between(df['date'], bottom, bottom + control_contrib, alpha=0.3,
                    label='Controls', color='brown')
    ax.plot(df['date'], df[target], 'ko-', markersize=4, label='Actual', linewidth=1.5)
    ax.set_ylabel('Quotes')
    ax.set_title(f'{PRODUCT_LABELS[target]} — Media Contribution Decomposition')
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)

axes[-1].set_xlabel('Month')
plt.tight_layout()
plt.savefig(figures_path / 'causal_media_decomposition.png', dpi=150)
plt.show()

In [ ]:
# ---------- Cross-product comparison table ----------
print('Ridge Coefficients — Media Channels (standardised scale)')
print('=' * 72)
rows = []
for ch in SATURATED_COLS:
    label = ch.replace('media_', '').replace('_saturated', '')
    row = {'Channel': label}
    for target in TARGET_COLS:
        bs = boot_summary[target]
        r = bs[bs['feature'] == ch].iloc[0]
        row[PRODUCT_LABELS[target]] = f"{r['coef']:>7.1f}  [{r['ci_lower']:>6.1f}, {r['ci_upper']:>6.1f}]"
    rows.append(row)

coef_table = pd.DataFrame(rows).set_index('Channel')
print(coef_table.to_string())

---
## Section 5 — Model 3: Elastic Net (Variable Selection Check)

Elastic Net combines L1 (lasso, variable selection) and L2 (ridge) penalties.  
We use it to check **which channels get zeroed out** — i.e., which ones Lasso deems uninformative.

In [ ]:
# ---------- Elastic Net per product ----------
enet_results = {}

for target in TARGET_COLS:
    y = df[target].values
    enet = ElasticNetCV(
        l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
        alphas=np.logspace(-3, 2, 50),
        cv=LeaveOneOut(),
        max_iter=10_000,
    )
    enet.fit(X_scaled, y)
    enet_results[target] = enet
    print(f'{PRODUCT_LABELS[target]:25s}  alpha={enet.alpha_:.4f}  '
          f'l1_ratio={enet.l1_ratio_:.2f}  '
          f'non-zero={np.sum(enet.coef_ != 0)}/{len(enet.coef_)}  '
          f'R²={r2_score(y, enet.predict(X_scaled)):.3f}')

In [ ]:
# ---------- Side-by-side coefficient comparison (OLS / Ridge / Elastic Net) ----------
for target in TARGET_COLS:
    print(f'\n{"=" * 72}')
    print(f'{PRODUCT_LABELS[target]} — Coefficient Comparison')
    print(f'{"=" * 72}')

    ols_coefs = ols_results[target].params[1:]  # skip constant
    ridge_coefs = ridge_results[target]['model'].coef_
    enet_coefs  = enet_results[target].coef_

    comp = pd.DataFrame({
        'Feature': FEATURE_COLS,
        'OLS':     ols_coefs.values,
        'Ridge':   ridge_coefs,
        'ElasticNet': enet_coefs,
    })
    comp['label'] = comp['Feature'].str.replace('media_', '').str.replace('_saturated', '').str.replace('_scaled', '')
    comp = comp[['label', 'OLS', 'Ridge', 'ElasticNet']]
    print(comp.to_string(index=False, float_format='{:>8.2f}'.format))

---
## Section 6 — Channel Effectiveness Metrics (Key Deliverable)

Translate Ridge coefficients back into **business metrics**:

- Marginal effect: additional quotes per \$1 000 extra spend.
- Cost per incremental quote.
- Saturation curves: expected response at various spend levels.
- Cross-product heatmap.

In [ ]:
# ===================================================================
# 6-A  MARGINAL EFFECTS  (dQuotes / d$1000)
# ===================================================================
# Chain rule:  dQuotes/dSpend = beta_ridge * (dSaturated/dAdstock) * (dAdstock/dSpend) / std_X
#
# For Hill:  dH/dx = alpha * K^alpha * x^(alpha-1) / (x^alpha + K^alpha)^2
# For geometric adstock:  dAdstock/dSpend ≈ 1/(1-lambda)  (steady-state gain)
#
# We evaluate the derivative at the MEAN adstocked spend.
# hill_derivative imported from src.features.transformations in cell-6

effectiveness_rows = []

for target in TARGET_COLS:
    model = ridge_results[target]['model']
    bs    = boot_summary[target]

    for i, ch in enumerate(MEDIA_CHANNELS):
        sat_col = SATURATED_COLS[i]
        feat_idx = FEATURE_COLS.index(sat_col)
        beta_std = model.coef_[feat_idx]

        # Scale factor to convert from standardised to original
        sat_std = df[sat_col].std()
        sat_mean = df[sat_col].mean()

        # Saturation derivative at mean adstock
        K     = SATURATION_PARAMS[ch]['K']
        alpha = SATURATION_PARAMS[ch]['alpha']
        adstock_mean = df[f'{ch}_adstock'].mean()
        dH_dx = float(hill_derivative(adstock_mean, K, alpha))

        # Adstock gain (steady-state)
        lam = DECAY_RATES[ch]
        adstock_gain = 1.0 / (1.0 - lam)

        # Marginal effect:  dQuotes / d$ = beta * (1/std_sat) * dH/dAdstock * adstock_gain
        if sat_std > 0:
            marginal_per_dollar = beta_std / sat_std * dH_dx * adstock_gain
        else:
            marginal_per_dollar = 0.0

        marginal_per_1000 = marginal_per_dollar * 1000

        # Cost per incremental quote
        cpiq = 1000 / marginal_per_1000 if abs(marginal_per_1000) > 1e-6 else np.inf

        # Total spend and total contribution
        total_spend = df[ch].sum()
        # Contribution = beta_std * (mean standardised feature)
        # We use the actual mean contribution across time
        contrib = float((X_scaled[:, feat_idx] * beta_std).sum())

        # Current saturation level (how saturated is the channel at current spend?)
        current_sat = float(df[sat_col].mean()) * 100  # Hill is [0,1]

        # Bootstrap CI for marginal effect
        boot_arr = np.array(boot_coefs[target])[:, feat_idx]  # bootstrap betas
        if sat_std > 0:
            boot_marginals = boot_arr / sat_std * dH_dx * adstock_gain * 1000
        else:
            boot_marginals = np.zeros(N_BOOT)
        ci_lo = float(np.percentile(boot_marginals, 5))
        ci_hi = float(np.percentile(boot_marginals, 95))

        effectiveness_rows.append({
            'product':     PRODUCT_LABELS[target],
            'channel':     ch.replace('media_', ''),
            'total_spend': total_spend,
            'ridge_coef_std': beta_std,
            'marginal_per_1000': marginal_per_1000,
            'marginal_ci_lo': ci_lo,
            'marginal_ci_hi': ci_hi,
            'cost_per_quote': cpiq,
            'total_contribution': contrib,
            'saturation_pct': current_sat,
        })

eff_df = pd.DataFrame(effectiveness_rows)

# Display per product
for target in TARGET_COLS:
    label = PRODUCT_LABELS[target]
    print(f'\n{"=" * 90}')
    print(f'  {label} — Channel Effectiveness')
    print(f'{"=" * 90}')
    sub = eff_df[eff_df['product'] == label].copy()
    sub = sub.sort_values('marginal_per_1000', ascending=False)
    display_cols = ['channel', 'total_spend', 'marginal_per_1000',
                    'marginal_ci_lo', 'marginal_ci_hi', 'cost_per_quote', 'saturation_pct']
    print(sub[display_cols].to_string(index=False, float_format='{:.2f}'.format))

In [ ]:
# ---------- Cross-product effectiveness heatmap ----------
pivot = eff_df.pivot_table(index='channel', columns='product',
                           values='marginal_per_1000')

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlGn', center=0, ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Incremental quotes per $1 000'})
ax.set_title('Cross-Product Media Effectiveness (quotes per $1 000 spend)', fontsize=13)
ax.set_ylabel('Channel')
ax.set_xlabel('Product Category')
plt.savefig(figures_path / 'causal_effectiveness_heatmap.png', dpi=150)
plt.show()

In [ ]:
# ---------- Saturation curves: response vs spend per channel, per product ----------
fig, axes = plt.subplots(len(TARGET_COLS), 1, figsize=(14, 5 * len(TARGET_COLS)))

saturation_curve_rows = []

for row, target in enumerate(TARGET_COLS):
    ax = axes[row]
    model = ridge_results[target]['model']

    for i, ch in enumerate(MEDIA_CHANNELS):
        sat_col  = SATURATED_COLS[i]
        feat_idx = FEATURE_COLS.index(sat_col)
        beta_std = model.coef_[feat_idx]
        sat_std  = df[sat_col].std()
        sat_mean_val = df[sat_col].mean()

        K     = SATURATION_PARAMS[ch]['K']
        alpha_h = SATURATION_PARAMS[ch]['alpha']
        lam   = DECAY_RATES[ch]

        # Spend grid: $0 to 2x current max
        max_spend = max(df[ch].max() * 2, 1000)
        spend_grid = np.linspace(0, max_spend, 200)

        # Adstock of a constant spend stream → steady-state adstock = spend / (1 - lambda)
        adstock_grid = spend_grid / (1 - lam)
        sat_grid = hill_saturation(adstock_grid, K, alpha_h)
        # Convert to incremental quotes via Ridge coefficient
        if sat_std > 0:
            quotes_grid = beta_std / sat_std * (sat_grid - sat_mean_val)
        else:
            quotes_grid = np.zeros_like(sat_grid)

        label = ch.replace('media_', '')
        ax.plot(spend_grid, quotes_grid, linewidth=2, label=label.title())

        # Save curve data
        for s, q in zip(spend_grid[::20], quotes_grid[::20]):
            saturation_curve_rows.append({
                'product': PRODUCT_LABELS[target],
                'channel': label,
                'spend': s,
                'incremental_quotes': q,
            })

    ax.set_xlabel('Monthly Spend ($)')
    ax.set_ylabel('Incremental Quotes (vs mean)')
    ax.set_title(f'{PRODUCT_LABELS[target]} — Saturation Curves')
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
    ax.axhline(0, color='grey', linestyle='--', linewidth=0.6)

plt.tight_layout()
plt.savefig(figures_path / 'causal_saturation_curves.png', dpi=150)
plt.show()

sat_curves_df = pd.DataFrame(saturation_curve_rows)

In [ ]:
# ---------- Efficiency frontier: spend share vs contribution share ----------
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for idx, target in enumerate(TARGET_COLS):
    ax = axes[idx]
    label = PRODUCT_LABELS[target]
    sub = eff_df[eff_df['product'] == label].copy()
    total_sp = sub['total_spend'].sum()
    # Use absolute contributions for share (positive only)
    sub['abs_contrib'] = sub['total_contribution'].clip(lower=0)
    total_c = sub['abs_contrib'].sum()
    sub['spend_share']  = sub['total_spend'] / total_sp * 100 if total_sp > 0 else 0
    sub['contrib_share'] = sub['abs_contrib'] / total_c * 100 if total_c > 0 else 0

    ax.scatter(sub['spend_share'], sub['contrib_share'], s=80, zorder=3)
    for _, r in sub.iterrows():
        ax.annotate(r['channel'], (r['spend_share'], r['contrib_share']),
                    textcoords='offset points', xytext=(6, 6), fontsize=8)
    # 45-degree line
    mn, mx = 0, max(sub['spend_share'].max(), sub['contrib_share'].max()) * 1.1
    ax.plot([0, mx], [0, mx], 'k--', linewidth=0.7, alpha=0.5)
    ax.set_xlabel('Spend Share (%)')
    ax.set_ylabel('Contribution Share (%)')
    ax.set_title(f'{label}')

fig.suptitle('Efficiency Frontier — Spend Share vs Contribution Share', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(figures_path / 'causal_efficiency_frontier.png', dpi=150)
plt.show()

---
## Section 7 — Robustness Checks

In [ ]:
# ===================================================================
# 7-A  SENSITIVITY TO ADSTOCK DECAY RATES (+/- 0.1)
# ===================================================================

decay_variants = {'low': -0.1, 'base': 0.0, 'high': +0.1}
sensitivity_decay = []

for variant_name, delta in decay_variants.items():
    df_tmp = df_raw.copy()
    df_tmp = df_tmp.sort_values('date').reset_index(drop=True)

    # Rebuild channel aggregation
    for group, cols in CHANNEL_MAP.items():
        existing = [c for c in cols if c in df_tmp.columns]
        df_tmp[f'media_{group}'] = df_tmp[existing].sum(axis=1) if existing else 0

    # Adstock with perturbed decay
    for ch in MEDIA_CHANNELS:
        lam = np.clip(DECAY_RATES[ch] + delta, 0.05, 0.95)
        df_tmp[f'{ch}_adstock'] = geometric_adstock(df_tmp[ch].fillna(0).values, lam)

    # Saturation (recompute K on perturbed adstock)
    for ch in MEDIA_CHANNELS:
        adstock_col = f'{ch}_adstock'
        nz = df_tmp[adstock_col][df_tmp[adstock_col] > 0]
        K = float(nz.median()) if len(nz) > 0 else 1.0
        df_tmp[f'{ch}_saturated'] = hill_saturation(df_tmp[adstock_col].values, K, HILL_ALPHA)

    # Controls
    df_tmp['sin_1'] = np.sin(2 * np.pi * df_tmp['month_num'] / 12)
    df_tmp['cos_1'] = np.cos(2 * np.pi * df_tmp['month_num'] / 12)
    df_tmp['avg_temp_mean_scaled'] = (df_tmp['avg_temp_mean'] - temp_mean) / temp_std
    df_tmp['is_pool_season'] = df_tmp['month_num'].isin([4,5,6,7,8]).astype(int)

    X_tmp = StandardScaler().fit_transform(df_tmp[FEATURE_COLS].fillna(0))

    for target in TARGET_COLS:
        y = df_tmp[target].values
        alpha_r = ridge_results[target]['best_alpha']
        m = Ridge(alpha=alpha_r).fit(X_tmp, y)
        for j, ch in enumerate(MEDIA_CHANNELS):
            sensitivity_decay.append({
                'variant': variant_name,
                'product': PRODUCT_LABELS[target],
                'channel': ch.replace('media_', ''),
                'coef': m.coef_[j],
            })

sens_decay_df = pd.DataFrame(sensitivity_decay)
pivot_sens = sens_decay_df.pivot_table(index=['product', 'channel'], columns='variant', values='coef')
pivot_sens = pivot_sens[['low', 'base', 'high']]
pivot_sens['sign_stable'] = ((pivot_sens['low'] > 0) & (pivot_sens['high'] > 0)) | \
                            ((pivot_sens['low'] < 0) & (pivot_sens['high'] < 0)) | \
                            ((pivot_sens['low'] == 0) & (pivot_sens['high'] == 0))

print('Adstock Decay Sensitivity (coefficient sign stability):')
print(pivot_sens.to_string(float_format='{:.2f}'.format))
print(f'\nSign-stable: {pivot_sens["sign_stable"].sum()}/{len(pivot_sens)}')

In [ ]:
# ===================================================================
# 7-B  SENSITIVITY TO SATURATION PARAMS (K and alpha)
# ===================================================================

sat_variants = {
    'K_low':   {'K_mult': 0.5, 'alpha': 2},
    'base':    {'K_mult': 1.0, 'alpha': 2},
    'K_high':  {'K_mult': 2.0, 'alpha': 2},
    'alpha_1': {'K_mult': 1.0, 'alpha': 1},
    'alpha_3': {'K_mult': 1.0, 'alpha': 3},
}

sensitivity_sat = []

for variant_name, params in sat_variants.items():
    df_tmp = df.copy()  # already has adstock cols from base
    for ch in MEDIA_CHANNELS:
        base_K = SATURATION_PARAMS[ch]['K']
        K_v = base_K * params['K_mult']
        a_v = params['alpha']
        df_tmp[f'{ch}_saturated'] = hill_saturation(df_tmp[f'{ch}_adstock'].values, K_v, a_v)

    X_tmp = StandardScaler().fit_transform(df_tmp[FEATURE_COLS].fillna(0))

    for target in TARGET_COLS:
        y = df_tmp[target].values
        alpha_r = ridge_results[target]['best_alpha']
        m = Ridge(alpha=alpha_r).fit(X_tmp, y)
        for j, ch in enumerate(MEDIA_CHANNELS):
            sensitivity_sat.append({
                'variant': variant_name,
                'product': PRODUCT_LABELS[target],
                'channel': ch.replace('media_', ''),
                'coef': m.coef_[j],
            })

sens_sat_df = pd.DataFrame(sensitivity_sat)
pivot_sat = sens_sat_df.pivot_table(index=['product', 'channel'], columns='variant', values='coef')
pivot_sat = pivot_sat[['K_low', 'base', 'K_high', 'alpha_1', 'alpha_3']]

print('Saturation Parameter Sensitivity:')
print(pivot_sat.to_string(float_format='{:.2f}'.format))

In [ ]:
# ===================================================================
# 7-C  LEAVE-ONE-OUT INFLUENCE ANALYSIS
# ===================================================================

loo_influence = []

for target in TARGET_COLS:
    alpha_r = ridge_results[target]['best_alpha']
    y = df[target].values
    full_coefs = ridge_results[target]['model'].coef_

    for drop_i in range(len(df)):
        mask = np.ones(len(df), dtype=bool)
        mask[drop_i] = False
        X_loo = X_scaled[mask]
        y_loo = y[mask]
        m = Ridge(alpha=alpha_r).fit(X_loo, y_loo)
        # Coefficient change
        for j, ch in enumerate(MEDIA_CHANNELS):
            loo_influence.append({
                'product': PRODUCT_LABELS[target],
                'dropped_month': str(df['date'].iloc[drop_i].date()),
                'channel': ch.replace('media_', ''),
                'coef_full': full_coefs[j],
                'coef_loo': m.coef_[j],
                'coef_change': m.coef_[j] - full_coefs[j],
            })

loo_df = pd.DataFrame(loo_influence)

# Summary: max absolute coefficient change per channel
loo_summary = loo_df.groupby(['product', 'channel']).agg(
    max_abs_change=('coef_change', lambda x: x.abs().max()),
    mean_abs_change=('coef_change', lambda x: x.abs().mean()),
).reset_index()

print('Leave-One-Out Influence (max & mean absolute coefficient change):')
print(loo_summary.to_string(index=False, float_format='{:.3f}'.format))

In [ ]:
# ===================================================================
# 7-D  PERMUTATION FEATURE IMPORTANCE
# ===================================================================

from sklearn.inspection import permutation_importance

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, target in enumerate(TARGET_COLS):
    ax = axes[idx]
    model = ridge_results[target]['model']
    y = df[target].values

    perm = permutation_importance(model, X_scaled, y, n_repeats=100,
                                  random_state=42, scoring='r2')
    perm_df = pd.DataFrame({
        'feature': FEATURE_COLS,
        'importance_mean': perm.importances_mean,
        'importance_std':  perm.importances_std,
    }).sort_values('importance_mean', ascending=True)

    # Plot only media features
    perm_media = perm_df[perm_df['feature'].str.contains('_saturated')].copy()
    perm_media['label'] = perm_media['feature'].str.replace('media_', '').str.replace('_saturated', '')

    ax.barh(perm_media['label'], perm_media['importance_mean'], alpha=0.8, color='teal')
    ax.errorbar(perm_media['importance_mean'], perm_media['label'],
                xerr=perm_media['importance_std'],
                fmt='none', ecolor='black', capsize=3)
    ax.set_xlabel('Mean R² Decrease')
    ax.set_title(f'{PRODUCT_LABELS[target]}')
    ax.axvline(0, color='grey', linestyle='--', linewidth=0.6)

axes[0].set_ylabel('Channel')
fig.suptitle('Permutation Feature Importance (media channels)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(figures_path / 'causal_permutation_importance.png', dpi=150)
plt.show()

In [ ]:
# ===================================================================
# 7-E  ROBUSTNESS SUMMARY MATRIX
# ===================================================================

# For each (product, channel): is the coefficient direction consistent
# across (a) adstock sensitivity, (b) saturation sensitivity, (c) LOO?

robustness_rows = []
for target in TARGET_COLS:
    label = PRODUCT_LABELS[target]
    for ch in MEDIA_CHANNELS:
        ch_short = ch.replace('media_', '')

        # Adstock sign stability
        adstock_sub = sens_decay_df[(sens_decay_df['product'] == label) &
                                    (sens_decay_df['channel'] == ch_short)]
        adstock_signs = np.sign(adstock_sub['coef'].values)
        adstock_stable = len(set(adstock_signs[adstock_signs != 0])) <= 1

        # Saturation sign stability
        sat_sub = sens_sat_df[(sens_sat_df['product'] == label) &
                              (sens_sat_df['channel'] == ch_short)]
        sat_signs = np.sign(sat_sub['coef'].values)
        sat_stable = len(set(sat_signs[sat_signs != 0])) <= 1

        # LOO: max change < 50% of base coef
        loo_sub = loo_summary[(loo_summary['product'] == label) &
                              (loo_summary['channel'] == ch_short)]
        base_coef = abs(ridge_results[target]['model'].coef_[
            FEATURE_COLS.index(f'{ch}_saturated')])
        loo_stable = True
        if len(loo_sub) > 0 and base_coef > 0:
            loo_stable = loo_sub['max_abs_change'].values[0] < 0.5 * base_coef

        # Bootstrap CI excludes zero?
        bs = boot_summary[target]
        bs_row = bs[bs['feature'] == f'{ch}_saturated'].iloc[0]
        ci_excludes_zero = (bs_row['ci_lower'] > 0) or (bs_row['ci_upper'] < 0)

        robustness_rows.append({
            'Product':  label,
            'Channel':  ch_short,
            'Adstock-stable':    'Yes' if adstock_stable else 'No',
            'Saturation-stable': 'Yes' if sat_stable else 'No',
            'LOO-stable':        'Yes' if loo_stable else 'No',
            'CI-excludes-0':     'Yes' if ci_excludes_zero else 'No',
        })

robust_df = pd.DataFrame(robustness_rows)
print('Robustness Summary Matrix:')
print(robust_df.to_string(index=False))

---
## Section 8 — Limitations & Causal Interpretation

### Small-sample caveats
- **Small N**: With a limited number of monthly observations and 11 parameters, statistical power is limited even with Ridge shrinkage.
- **Wide confidence intervals** — many channels' CIs cross zero. Absence of significance ≠ absence of effect.
- **Data gaps** — missing months between fiscal years may affect adstock carry-over for channels active before gaps.

### Identification assumptions
1. **Conditional exogeneity** — after controlling for seasonality and weather, the remaining variation in media spend is unrelated to unobserved demand shocks. Violated if, for example, the marketing team increased spend *because* they observed rising demand from another source.
2. **No simultaneity** — quotes cause no immediate spend adjustment within the same month.
3. **Correct functional form** — geometric adstock and Hill saturation are assumed; the true response shape is unknown.
4. **Stable coefficients** — the relationship between spend and quotes is assumed constant across the observation window.

### What these results CAN vs CANNOT say
| CAN | CANNOT |
|-----|--------|
| Identify which channels have the strongest *association* with quotes after controlling for seasonality | Prove strict causation without experimental variation |
| Provide directional guidance on channel efficiency | Give precise point estimates (CIs are wide) |
| Flag channels that are likely saturated | Rule out channels whose CIs include zero |

### Recommendations for stronger evidence
- **More data**: extending to 36+ months of continuous history would reduce uncertainty substantially.
- **Geo-experiments**: randomised regional spend variation would provide stronger causal identification.
- **Weekly granularity**: 80+ weekly observations would allow richer lag structures.

---
## Section 9 — Bridge to Optimization

### Deliverables for Notebook 07
1. **Marginal effects per product** — `media_effectiveness_results.csv`
2. **Saturation curves** — `saturation_curves.csv` (spend → incremental quotes, per product per channel)
3. **Ridge models** — retained in memory; can be pickled if needed.

### How this feeds into budget optimization
- The optimizer will maximise total quotes (or a weighted combination across products) subject to a total budget constraint.
- The saturation curves tell the optimizer *how much* each additional dollar buys in each channel.
- Running separate models per product lets the optimizer balance the portfolio across product lines.

In [ ]:
# ---------- Save outputs ----------

# 1. Effectiveness table
eff_df.to_csv(processed_path / 'media_effectiveness_results.csv', index=False)
print(f'Saved: {processed_path / "media_effectiveness_results.csv"}')

# 2. Saturation curves
sat_curves_df.to_csv(processed_path / 'saturation_curves.csv', index=False)
print(f'Saved: {processed_path / "saturation_curves.csv"}')

# 3. Robustness summary
robust_df.to_csv(processed_path / 'robustness_summary.csv', index=False)
print(f'Saved: {processed_path / "robustness_summary.csv"}')

# 4. Model parameters (for reproducibility)
model_params = {
    'decay_rates': {k.replace('media_', ''): v for k, v in DECAY_RATES.items()},
    'saturation_params': {k.replace('media_', ''): v for k, v in SATURATION_PARAMS.items()},
    'ridge_alphas': {PRODUCT_LABELS[t]: float(ridge_results[t]['best_alpha']) for t in TARGET_COLS},
    'ridge_r2': {PRODUCT_LABELS[t]: float(ridge_results[t]['r2']) for t in TARGET_COLS},
    'feature_cols': FEATURE_COLS,
    'media_channels': [c.replace('media_', '') for c in MEDIA_CHANNELS],
}
with open(processed_path / 'causal_model_params.json', 'w') as f:
    json.dump(model_params, f, indent=2, default=str)
print(f'Saved: {processed_path / "causal_model_params.json"}')

In [ ]:
# ---------- Final summary ----------
print('=' * 72)
print('  NOTEBOOK 06 — CAUSAL INFERENCE: COMPLETE')
print('=' * 72)
print()
print('Models fitted per product (Ridge, LOOCV):')
for target in TARGET_COLS:
    r = ridge_results[target]
    print(f'  {PRODUCT_LABELS[target]:25s}  alpha={r["best_alpha"]:>8.2f}  '
          f'R²={r["r2"]:.3f}  MAE={r["mae"]:.1f}')
print()
print('Output files:')
print(f'  data/processed/media_effectiveness_results.csv')
print(f'  data/processed/saturation_curves.csv')
print(f'  data/processed/robustness_summary.csv')
print(f'  data/processed/causal_model_params.json')
print()
print('Figures saved to reports/figures/:')
for f in sorted(figures_path.glob('causal_*.png')):
    print(f'  {f.name}')
print()
print('Next step: notebooks/07_budget_optimization.ipynb')